# 06 · Guardrails and approval

Notebook 03 ended with the agent refunding $250 for the late espresso machine with nobody's approval.
Policy P5 says any refund over $100 needs a supervisor, the model had read it, and it refunded anyway. A
rule in a prompt is advice; a **guardrail** is a check in code that the model cannot skip.

**You will learn**

- the guardrails ShipFast already has, and where each lives
- how to pause a LangGraph agent before a risky action with `interrupt()`
- how a person approves or rejects, and how the run resumes
- why the pause needs a checkpointer

In [1]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

from agent_contracts.harness import verify_model
from agent_contracts.shipfast import tools as shipfast
from lg_agents.llm import make_llm
from lg_agents.prompts import SPECIALIST
from lg_agents.tools import shipfast_tools

verify_model()
llm, tools = make_llm(), shipfast_tools()
LATE = "Order A1003 is more than a week late. Please cancel it and refund me."

## Guardrails so far

| Guardrail | Where it lives | What it stops |
|---|---|---|
| `issue_refund` refuses amounts above the order total | the shared tool | a made-up refund amount (notebook 02) |
| `recursion_limit` | the graph runtime | a run that never stops (notebook 03) |
| the ledger check | ShipFast's test runner | a reply or result that claims a refund that never happened |
| **approval for refunds over $100** | **the graph: this notebook** | money moving without a person's yes |

The first three check things after the fact or refuse bad input. Approval is different: the run must
**stop and wait** for a person, possibly for hours, and then carry on exactly where it was.

## Pausing with `interrupt()`

LangGraph's `interrupt(payload)` stops the graph inside a node, saves the state through the checkpointer,
and hands `payload` to the caller. Later the caller resumes with `Command(resume=value)`; the node runs
again from the top and this time `interrupt()` returns `value`.

To pause only before refunds over $100, the gate wraps every tool call as **middleware**. LangChain also
ships `HumanInTheLoopMiddleware`, which pauses before every call to a named tool; that would also stop
the $45 refund, which needs no approval, so ShipFast uses a small custom gate. Here is all of it:

In [2]:
import inspect

import lg_agents.approval

print(inspect.getsource(lg_agents.approval))

"""ShipFast's approval gate (SPEC.md section 3, policy P5) as LangChain agent middleware.

Before any `issue_refund` call over $100.00 runs, the graph pauses with LangGraph's `interrupt()` and
waits for a supervisor's decision, supplied later with `Command(resume="approve" | "reject")`. The pause
is in the framework, not in the prompt and not in the shared tool. Needs a checkpointer, because the
paused state must be saved to resume it.

Approve: the refund runs. Reject: the refund does not run; the model gets a rejection it must act on by
escalating to a human.
"""

from __future__ import annotations

import json

from langchain.agents.middleware import wrap_tool_call
from langchain_core.messages import ToolMessage
from langgraph.types import interrupt

from agent_contracts.shipfast.tools import APPROVAL_THRESHOLD  # policy P5, shared by all builds


def _amount(args: dict) -> float | None:
    try:
        return float(args.get("amount"))
    except (TypeError, ValueError):
        re

In [3]:
agent = create_agent(llm, tools, system_prompt=SPECIALIST, middleware=[lg_agents.approval.refund_approval],
                     checkpointer=InMemorySaver())


def ledger():
    return [(r["order_id"], r["amount"]) for r in shipfast.refunds()] or "no refund"

## Under $100: no pause

The lost running shoes (A1002) are a $45 refund. The gate lets it straight through.

In [4]:
shipfast.reset()
state = agent.invoke({"messages": [HumanMessage("My order A1002 never arrived and tracking says it's lost. I want a refund.")]},
                     {"configurable": {"thread_id": "S2"}})
print("paused:", "__interrupt__" in state)
print("ledger:", ledger())
print("reply: ", state["messages"][-1].content.strip()[:160])

paused: False
ledger: [('A1002', 45.0)]
reply:  Your refund for order A1002 has been issued. The total amount of $45.00 will be refunded to your original payment method. Thank you for your patience.


## Over $100: the run stops

The espresso machine (A1003) is $250. The run returns early with an `__interrupt__` entry carrying the
gate's payload. Nothing has been refunded, and the saved state shows the graph waiting at the `tools`
node.

In [5]:
shipfast.reset()
approve_thread = {"configurable": {"thread_id": "S3a"}}
state = agent.invoke({"messages": [HumanMessage(LATE)]}, approve_thread)

print("paused with:", state["__interrupt__"][0].value)
print("ledger while paused:", ledger())
print("waiting at:", agent.get_state(approve_thread).next)

paused with: {'type': 'refund_approval', 'order_id': 'A1003', 'amount': 250.0, 'reason': 'The order is delayed beyond the ETA and the customer is requesting a refund.', 'question': "Approve a refund of $250.00? Resume with 'approve' or 'reject'."}
ledger while paused: no refund
waiting at: ('tools',)


## The supervisor approves

Resuming with `Command(resume="approve")` re-runs the paused tool call, which now goes through.

In [6]:
state = agent.invoke(Command(resume="approve"), approve_thread)
print("ledger:", ledger())
print("reply: ", state["messages"][-1].content.strip()[:200])

ledger: [('A1003', 250.0)]
reply:  Your refund for order A1003 has been issued. You will receive a full refund of $250.00. Thank you for your patience and understanding.


## The supervisor rejects

Same message, new thread, and this time `reject`. The gate does not run the refund; it hands the model a
rejection telling it to escalate. Policy P5 says a rejected refund goes to a human agent.

In [7]:
shipfast.reset()
reject_thread = {"configurable": {"thread_id": "S3b"}}
agent.invoke({"messages": [HumanMessage(LATE)]}, reject_thread)
state = agent.invoke(Command(resume="reject"), reject_thread)
print("ledger: ", ledger())
print("tickets:", [t["ticket_id"] for t in shipfast.tickets()])
print("tools:  ", shipfast.tool_calls())
print("reply:  ", state["messages"][-1].content.strip()[:200])

ledger:  no refund
tickets: ['TCK-A1003']
tools:   ['lookup_order', 'track_shipment', 'search_policy', 'escalate_to_human']
reply:   The refund for your order (A1003) has been rejected by the supervisor. A specialist has been notified and will follow up with you. You can check the status of your ticket using the reference number TC


Two things make this a real guardrail rather than a suggestion. The refund cannot happen without the
`approve` value, because the tool is not called until then. And the escalation after a rejection, which
*is* left to the model, is checked by ShipFast's tests: scenario S3b expects `escalated`.

## Why the checkpointer matters

`interrupt()` only works with a checkpointer: the paused state has to be stored somewhere so a later
call can pick it up. With `InMemorySaver` that later call must come from this same process. With a
database checkpointer (notebook 04) the supervisor could approve from a different process, hours later.

## How the others do this

- **CrewAI** pauses inside a **Flow**: a step decorated with `@human_feedback` stops for a person's
  input, and `@persist` saves the flow's state so it can resume. The pause is around a flow step, so the
  refund decision has to be its own step rather than a tool call deep inside an agent.
- **Strands** can interrupt from a hook on `BeforeToolCallEvent`, which fires before each tool call; the
  hook can check the refund amount and pause, much like this gate.

## Takeaways

- A prompt asks; a guardrail enforces. Put money-moving checks in code.
- `interrupt()` pauses a LangGraph run and `Command(resume=...)` continues it from the same spot.
- Middleware can wrap every tool call, so one small gate covers the whole agent.
- Next: **07 · Tracing and eval**, measuring runs the same way in all three frameworks.